# Train the Market-1501 Strong Baseline on a Colab GPU

This notebook reproduces [`person-reid-market1501`](https://github.com/vardhjain/person-reid-market1501)
end to end on a **free Colab GPU**:

1. Install the package
2. Download Market-1501
3. Train the ResNet-50 + BNNeck strong baseline (~40 min on a T4)
4. Evaluate (CMC / mAP + k-reciprocal re-ranking)
5. Generate Grad-CAM and ranked-result visualizations
6. Download the trained checkpoint

> **Before you start:** enable the GPU via *Runtime → Change runtime type → T4 GPU*.


## 1. Verify the GPU


In [ ]:
!nvidia-smi


## 2. Install the package


In [ ]:
!git clone --depth 1 https://github.com/vardhjain/person-reid-market1501.git
%cd person-reid-market1501
!pip install -q -e .


## 3. Download Market-1501

The dataset is pulled from Kaggle via `kagglehub`. Its root is exported as
`REID_DATA_ROOT`, which every CLI reads as a fallback when `--data-root` is omitted.


In [ ]:
import os
from pathlib import Path

import kagglehub

download_path = Path(kagglehub.dataset_download("pengcw1/market-1501"))
canonical = download_path / "Market-1501-v15.09.15"
data_root = canonical if canonical.is_dir() else download_path
os.environ["REID_DATA_ROOT"] = str(data_root)
print("Market-1501 data root:", data_root)


## 4. Train the strong baseline

Checkpoints are written to `outputs/`; `best.pth` is the best-by-mAP checkpoint.
Add `--max-epochs 5` for a quick smoke run first if you like.


In [ ]:
!python scripts/train.py --config configs/market1501_strong_baseline.yaml --device cuda --output-dir outputs


## 5. Evaluate (CMC / mAP + k-reciprocal re-ranking)


In [ ]:
!python scripts/evaluate.py --config configs/market1501_strong_baseline.yaml --weights outputs/best.pth --device cuda --rerank


## 6. Visualize (Grad-CAM + ranked results)


In [ ]:
!python scripts/visualize.py --config configs/market1501_strong_baseline.yaml --weights outputs/best.pth --device cuda --output outputs/viz --num-gradcam 5


## 7. Save your results

Download the trained checkpoint, then copy the printed **mAP / Rank-1 / Rank-5**
numbers into the README *Results* table.


In [ ]:
from google.colab import files

files.download("outputs/best.pth")
